In [ ]:
# Input: MD topology + trajectory for IF/OF conformations
# Output: PDF plots of transporter domain shifts
import MDAnalysis as mda
import MDAnalysis.analysis.rms
from MDAnalysis.analysis import align
import matplotlib.pyplot as plt
import prolif
import prolif.plotting.network
import seaborn as sns
import numpy as np

In [ ]:
rmsds = []
rmsds_prot = []
rmsds_sod = []

files = [('traj_if_0.xtc', 'structure_if_0.gro'),
        ('traj_if_1.xtc', 'structure_if_1.gro'),
        ('traj_if_2.xtc', 'structure_if_2.gro')]
pans_IF = []
cores_IF = []
diffs_IF = []
for trj_file, tp_file in files:
    u = mda.Universe(tp_file, trj_file)
    u_ref = mda.Universe(tp_file, trj_file)
    u_ref.trajectory[0]

    aligner = mda.analysis.align.AlignTraj(u, u_ref, select='(resname POPG or resname POPE and not name H*)', in_memory=True)
    aligner.run()

    for ts in u.trajectory:
        panel = u.select_atoms('backbone and ((resid 1 to 24) or (resid 133 to 183))')
        core = u.select_atoms('backbone and not ((resid 1 to 24) or (resid 133 to 183))')
        memb = u.select_atoms('(resname POPG or resname POPE) and not name H*')
        pancom = panel.center_of_mass()
        corecom = core.center_of_mass()
        membcom = memb.center_of_mass()

        pans_IF.append(pancom[2] - membcom[2])
        cores_IF.append(corecom[2] - membcom[2])
        diffs_IF.append(pancom[2] - corecom[2])

    plt.hist(pans_IF, label='Panel COM relative to membrane COM', color='red', alpha=0.4, bins=50)
    plt.hist(cores_IF, label='Core COM relative to membrane COM', color='blue', alpha=0.4, bins=50)
plt.show()
plt.close()

files = [('traj_of_0.xtc', 'structure_of_0.gro'),
        ('traj_of_1.xtc', 'structure_of_1.gro'),
        ('traj_of_2.xtc', 'structure_of_2.gro')]

pans_OF = []
cores_OF = []
diffs_OF = []
for trj_file, tp_file in files:
    u = mda.Universe(tp_file, trj_file)
    u_ref = mda.Universe(tp_file, trj_file)
    u_ref.trajectory[0]
    
    aligner = mda.analysis.align.AlignTraj(u, u_ref, select='(resname POPG or resname POPE and not name H*)', in_memory=True)
    aligner.run()

    for ts in u.trajectory:
        panel = u.select_atoms('backbone and ((resid 1 to 24) or (resid 133 to 183))')
        core = u.select_atoms('backbone and not ((resid 1 to 24) or (resid 133 to 183))')
        memb = u.select_atoms('(resname POPG or resname POPE) and not name H*')
        pancom = panel.center_of_mass()
        corecom = core.center_of_mass()
        membcom = memb.center_of_mass()

        pans_OF.append(pancom[2] - membcom[2])
        cores_OF.append(corecom[2] - membcom[2])
        diffs_OF.append(pancom[2] - corecom[2])

    plt.hist(pans_OF, label='Panel COM relative to membrane COM', color='red', alpha=0.4, bins=50)
    plt.hist(cores_OF, label='Core COM relative to membrane COM', color='blue', alpha=0.4, bins=50)
plt.show()
plt.close()

In [ ]:
import matplotlib.gridspec as gridspec

def start_vplot(row, col, wspace=0.5, hspace=0.5, figsize=(16, 12), height_ratios='', width_ratios='', org='1d'):
    axs = []
    fig = plt.figure(figsize=figsize)
    if not height_ratios:
        height_ratios = [1] * row
    if not width_ratios:
        width_ratios = [1] * col
    gs = gridspec.GridSpec(row, col, height_ratios=height_ratios, width_ratios=width_ratios)
    gs.update(wspace=wspace, hspace=hspace)
    if org == '1d':
        for r in range(0, row):
            for c in range(0, col):
                axs.append(fig.add_subplot(gs[r, c]))
    if org == '2d':
        for r in range(0, row):
            cs = []
            for c in range(0, col):
                cs.append(fig.add_subplot(gs[r, c]))
            axs.append(cs)
    return fig, axs


In [ ]:
from scipy.stats import gaussian_kde
from scipy.signal import fftconvolve
import numpy as np
fig, axs = start_vplot(2, 2, figsize=(8, 4), org='2d', hspace=0.5, wspace=0.2)
kernel_pts = np.linspace(-10, 10, num=10000)
dx = kernel_pts[1] - kernel_pts[0]

core_OF_kde = gaussian_kde(np.array(cores_OF).flatten())
core_IF_kde = gaussian_kde(np.array(cores_IF).flatten())

f = core_OF_kde(kernel_pts)
g = core_IF_kde(kernel_pts)

# distribution of X - Y
core_diff = fftconvolve(f, g[::-1], mode='same') * dx

pan_OF_kde = gaussian_kde(np.array(pans_OF).flatten())
pan_IF_kde = gaussian_kde(np.array(pans_IF).flatten())

f2 = pan_OF_kde(kernel_pts)
g2 = pan_IF_kde(kernel_pts)

pan_diff = fftconvolve(f2, g2[::-1], mode='same') * dx

diff_OF_kde = gaussian_kde(np.array(diffs_OF).flatten())
diff_IF_kde = gaussian_kde(np.array(diffs_IF).flatten())

f = diff_OF_kde(kernel_pts)
g = diff_IF_kde(kernel_pts)

# distribution of X - Y
diff_diff = fftconvolve(f, g[::-1], mode='same') * dx


axs[0][0].plot(kernel_pts, core_diff, label='Core', color='blue')

axs[0][0].plot(kernel_pts, pan_diff, label='Panel', color='red')
axs[0][0].set_xlabel(r'$\Delta Z = Z_{OF} - Z_{IF}$, relative to membrane ($\AA$)')
axs[0][0].set_ylim(0, 0.3)
axs[0][0].legend(frameon=False, loc=1)
axs[0][1].plot(kernel_pts, diff_diff, label='panel', color='purple')
axs[1][1].plot(cores_IF[:1000], color='blue', alpha=0.8)
axs[1][1].plot(pans_IF[:1000], color='red', alpha=0.8)
axs[0][1].set_xlabel(r'$\Delta Z = Z_{OF} - Z_{IF}$, panel relative to core ($\AA$)')
axs[1][0].set_ylim(-10, 10)
axs[1][0].plot(cores_OF[:1000], color='blue', alpha=0.8)
axs[1][0].plot(pans_OF[:1000], color='red', alpha=0.8)
axs[1][1].set_ylim(-10, 10)
axs[1][0].set_xlabel(r'Time (ns)')
axs[1][1].set_xlabel(r'Time (ns)')
axs[1][1].set_ylabel(r'$Z_{IF}$ ($\AA$)', labelpad=-5)
axs[1][0].set_ylabel(r'$Z_{OF}$ ($\AA$)', labelpad=-5)
plt.savefig('domain_shifts.pdf')
plt.show()
plt.close()
